# 03 — Bottleneck Analysis

**Where do We Lose? (time and money)** — Process Bottleneck Detection & Optimization System

This notebook builds the project's central deliverable for Objective 2: a
single **ranked bottleneck table**, combining processing time, waiting
time, volume, rework rate and resource utilization into one documented
`bottleneck_score` per stage. All scoring logic lives in
`src/process_mining/bottleneck.py` — this notebook visualizes and
interprets its output, it doesn't recompute anything itself.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.data.generate_data import STAGE_ORDER
from src.process_mining.event_log import load_event_log
from src.process_mining.bottleneck import (
    calculate_stage_metrics, calculate_bottleneck_score, rank_bottlenecks,
    build_bottleneck_table, DEFAULT_WEIGHTS,
)

pd.set_option("display.max_columns", None)
px.defaults.template = "plotly_white"

el = load_event_log("../data/raw/event_log.csv")
print(f"{len(el):,} events across {el['process_id'].nunique():,} processes")

385,523 events across 50,000 processes


## Stage Metrics

In [2]:
metrics = calculate_stage_metrics(el)
metrics.round(3)

,stage,volume,avg_processing_time,median_processing_time,p90_processing_time,avg_waiting_time,median_waiting_time,p90_waiting_time,rework_rate,n_employees,resource_utilization
0,Document Validation,59245,5.233,5.332,5.332,22.809,15.346,62.942,0.156,119,0.756
1,Credit Assessment,51058,26.241,26.447,26.447,36.147,22.800,87.176,0.048,155,0.769
2,Risk Review,46519,8.855,9.057,9.057,25.534,15.325,64.167,0.049,142,0.802
3,Approval,40890,1.935,1.957,1.957,7.499,0.998,15.734,0.021,95,0.786
4,Contract Generation,40857,4.334,4.390,4.390,17.886,14.938,63.214,0.020,139,0.811
5,Disbursement,40668,2.245,2.269,2.269,8.646,1.215,16.307,0.015,119,0.820


**Business Interpretation.** Before any scoring, the raw numbers already
tell most of the story: Credit Assessment has both the longest processing
time (~5x Document Validation's) and among the longest waiting time of any
stage, while also carrying real volume (>51,000 occurrences). Document
Validation has by far the highest rework rate. No single column identifies
"the" bottleneck on its own — that's exactly why a combined score is
useful.

## Bottleneck Score

In [3]:
print("Weights used (see bottleneck.py for the rationale):")
for k, v in DEFAULT_WEIGHTS.items():
    print(f"  {k:16s} {v:.0%}")

table = build_bottleneck_table(el)
table.round(3)

Weights used (see bottleneck.py for the rationale):
  processing_time  30%
  waiting_time     30%
  volume           20%
  rework_rate      10%
  utilization      10%


,stage,volume,avg_processing_time,median_processing_time,p90_processing_time,avg_waiting_time,median_waiting_time,p90_waiting_time,rework_rate,n_employees,resource_utilization,bottleneck_score,rank
0,Credit Assessment,51058,26.241,26.447,26.447,36.147,22.800,87.176,0.048,155,0.769,0.756,1
1,Document Validation,59245,5.233,5.332,5.332,22.809,15.346,62.942,0.156,119,0.756,0.501,2
2,Risk Review,46519,8.855,9.057,9.057,25.534,15.325,64.167,0.049,142,0.802,0.434,3
3,Contract Generation,40857,4.334,4.390,4.390,17.886,14.938,63.214,0.020,139,0.811,0.230,4
4,Disbursement,40668,2.245,2.269,2.269,8.646,1.215,16.307,0.015,119,0.820,0.116,5
5,Approval,40890,1.935,1.957,1.957,7.499,0.998,15.734,0.021,95,0.786,0.054,6


In [4]:
fig = px.bar(table.sort_values("bottleneck_score"), x="bottleneck_score", y="stage", orientation="h",
             text_auto=".2f", title="Bottleneck Score by Stage (weighted, normalized 0-1)",
             category_orders={"stage": table.sort_values("bottleneck_score")["stage"].tolist()})
fig.show()

**Business Interpretation — Top 3 Bottlenecks.**

1. **Credit Assessment** (score ≈ top-ranked) — the combination of the
   longest processing time in the process *and* substantial waiting time
   makes this the single largest source of cycle-time loss. It affects
   essentially every application that isn't rejected earlier.
2. **Document Validation** — driven less by raw duration and more by its
   very high rework rate (documents get sent back for correction far more
   than at any other stage) and the fact that, being the first stage, it
   touches 100% of applications.
3. **Risk Review** — the second-longest waiting time in the process; risk
   and large-loan-amount reviews compound here similarly to (but less
   severely than) Credit Assessment.

Approval, Contract Generation and Disbursement all score far lower — they
are comparatively fast, low-rework, and not meaningfully queued.

## Score Components (why each stage ranks where it does)

In [5]:
norm = table.copy()
for col, source in [
    ("processing_time", "avg_processing_time"), ("waiting_time", "avg_waiting_time"),
    ("volume", "volume"), ("rework_rate", "rework_rate"), ("utilization", "resource_utilization"),
]:
    lo, hi = table[source].min(), table[source].max()
    norm[col] = 0.0 if hi == lo else (table[source] - lo) / (hi - lo)

components = norm.melt(id_vars="stage", value_vars=list(DEFAULT_WEIGHTS.keys()),
                        var_name="component", value_name="normalized_value")
components["weighted_contribution"] = components.apply(
    lambda r: r["normalized_value"] * DEFAULT_WEIGHTS[r["component"]], axis=1
)

fig = px.bar(components, x="stage", y="weighted_contribution", color="component",
             title="Bottleneck Score Decomposition by Component",
             category_orders={"stage": STAGE_ORDER})
fig.show()

**Business Interpretation.** The decomposition shows *why* each stage
scores as it does, not just the final number: Credit Assessment's score is
dominated by processing time and waiting time, together explaining most of
its total — exactly what an operations manager needs to know before
deciding what to fix (more staff / faster decisioning tools, not a
documentation-quality initiative). Document Validation's contribution comes
disproportionately from `rework_rate` and `volume` rather than raw
duration — a different, cheaper class of fix (see the process-mining
notebook's finding on missing documents).

## Why a Weighted Sum, Not the Spec's Illustrative Product

In [6]:
# For comparison: the project spec's illustrative formula multiplies the
# three normalized factors together. Shown here purely to demonstrate why
# it was not used as the final formula (see bottleneck.py docstring for the
# full rationale) - a product is dominated by whichever factor is closest
# to 0, even when the other factors are high.
product_score = norm["waiting_time"] * norm["volume"] * norm["rework_rate"]
comparison = pd.DataFrame({
    "stage": table["stage"],
    "weighted_sum_score": table["bottleneck_score"],
    "illustrative_product_score": product_score,
}).sort_values("weighted_sum_score", ascending=False)
comparison.round(3)

,stage,weighted_sum_score,illustrative_product_score
0,Credit Assessment,0.756,0.130
1,Document Validation,0.501,0.534
2,Risk Review,0.434,0.048
3,Contract Generation,0.230,0.000
4,Disbursement,0.116,0.000
5,Approval,0.054,0.000


**Business Interpretation.** Under the pure-product formula, Contract
Generation and Disbursement's near-zero `rework_rate` factor collapses
their product score toward zero regardless of their other factors — plausible
here since they *are* genuinely low-impact stages, but the product formula
would behave the same way (mask a real bottleneck) for any stage whose
rework rate happened to normalize near zero even if its processing or
waiting time were high. The weighted sum used as the project's actual score
doesn't have this failure mode: a stage that is slow but has little rework
still scores appropriately high on its dominant factors.

## Summary

| Rank | Stage | Score | Primary driver |
|---|---|---|---|
| 1 | Credit Assessment | highest | processing time + waiting time |
| 2 | Document Validation | 2nd | rework rate + volume |
| 3 | Risk Review | 3rd | waiting time |

**Recommended focus, in priority order:**

1. **Credit Assessment** — the highest-leverage fix is capacity: more
   analysts or faster decisioning tooling would directly cut both
   processing and waiting time for the single largest share of total
   cycle time in the process (quantified in the resource-optimization
   notebook).
2. **Document Validation** — a process-quality fix (reducing missing
   documents at intake, e.g. clearer upload requirements on digital
   channels) would cut rework rather than requiring more headcount.
3. **Risk Review** — a smaller version of the Credit Assessment problem;
   worth a secondary capacity increase once Credit Assessment is addressed.

This ranked table is the direct input to the resource-allocation
optimization built in a later phase of this project.